In [1]:
import glob
import os
import pandas as pd
import numpy as np
import re

In [ ]:
def is_header_row(row) -> bool:
    return any(
        str(cell).strip().lower() in ['nombre', 'posición', 'pos']
        for cell in row
    )

def is_section_title(row) -> bool:
    pattern = re.compile(r"(catchers|pitchers|infielders|outfielders)", re.IGNORECASE)
    return any(pattern.fullmatch(str(cell).strip()) for cell in row)

def extract_table_sections_from_excel(filepath: str, preview_rows: int = 5) -> List[pd.DataFrame]:
    cols = ['id','raw_name','position']
    df_raw = pd.read_excel(filepath, header=None)
    start_rows = []
    title_rows = []

    for i in range(len(df_raw)):
        row = df_raw.iloc[i]
        if is_header_row(row) or is_section_title(row):
            title = next((str(cell).strip().lower() for cell in row if str(cell).strip().lower() in ['catchers', 'pitchers', 'infielders', 'outfielders']), None)
            start_rows.append(i)
            title_rows.append(title)

    tables = []
    for idx, start in enumerate(start_rows):
        end = start_rows[idx + 1] if idx + 1 < len(start_rows) else None
        section = df_raw.iloc[start:end]
        section.columns = section.iloc[0]
        section = section[1:]  # remove header row
        section.dropna(axis=1,how='all',inplace=True)
        section.dropna(axis=0,how='all',inplace=True)
        if len(section.columns):
            section.columns = [cols[i] if i < 3 else 'other' for i in range(len(section.columns)) ]
            section = section[['id', 'raw_name', 'position']]
        section = section.reset_index(drop=True)
        section['title'] = title_rows[idx]
        tables.append(section.head(preview_rows))

    return tables

In [123]:
import pdfplumber

def extract_pdf_tables_with_titles(pdf_path: str) -> pd.DataFrame:
    section_titles = ['CATCHER', 'INFIELD', 'OUTFIELD', 'PITCHER']
    title = None
    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text().split('\n')
            for line in text:
                for t in section_titles:
                    if t in line.upper():
                        title = t.capitalize()

            tables = page.extract_tables(table_settings={"vertical_strategy": "lines", 
                                               "horizontal_strategy": "text", 
                                               "snap_tolerance": 4,})
            # for table in tables:
            #     for row in table:
            #         if not row or len(row) < 4:
            #             continue
            #         id_candidate = row[0].strip() if row[0] else ""
            #         name_candidate = row[1].strip() if len(row) > 1 and row[1] else ""
            #         pos_candidate = row[2].strip() if len(row) > 2 and row[2] else ""

            #         if id_candidate.isdigit() and name_candidate:
            #             rows.append({
            #                 'id': id_candidate,
            #                 'raw_name': name_candidate,
            #                 'position': pos_candidate,
            #                 'title': title
            #             })

    return pd.DataFrame(tables).T

In [124]:
def extract_and_preview_documents(zip_path: str, extract_to: str = "temp_docs") -> dict:
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    content = {}
    for filename in os.listdir(extract_to):
        filepath = os.path.join(extract_to, filename)

        # if filename.lower().endswith(('.xlsx', '.xls')):
        #     try:
        #         tables = extract_table_sections_from_excel(filepath)
        #         content[filename] = tables
        #     except Exception as e:
        #         content[filename] = [f"Error al leer Excel: {e}"]

        if filename.lower().endswith('.pdf'):
            #try:
                text_preview = extract_pdf_tables_with_titles(filepath)
                content[filename] = [text_preview]
            #except Exception as e:
            #    content[filename] = [f"Error al leer PDF: {e}"]

        #else:
        #    content[filename] = ["Tipo de archivo no soportado"]
    return content

In [125]:
preview = extract_and_preview_documents("roosters.zip")

for filename, sections in preview.items():
    print(f"--- {filename} ---")
    for i, section in enumerate(sections):
        print(f"[Sección {i+1}]")
        print(section)
        print("\n")
    print("\n" + "-"*80 + "\n")

--- ROSTER DUR 10-JULIO-2025.pdf ---
[Sección 1]
                                         0  \
0   [27, NO NAC, D/Z, 110, 1.83, 8-Jul-00]   
1                             [, , , , , ]   
2         [14, , D/D, 95, 1.85, 14-May-92]   
3                             [, , , , , ]   
4       [4, NOV, D/D, 92, 1.78, 15-Jun-02]   
5                                     None   
6                                     None   
7                                     None   
8                                     None   
9                                     None   
10                                    None   
11                                    None   
12                                    None   
13                                    None   
14                                    None   
15                                    None   
16                                    None   
17                                    None   
18                                    None   
19                             

['Tipo de archivo no soportado']